# Task 1 — LoRA adaptation of small generalist VLMs

**Owner:** Shanmuga Perumal Balamurugan  
**Hypothesis (H1):** LoRA-adapted 3B generalists approach specialist parity on ScreenSpot-V2 (within ±3 pp) but retain a 10-15 pp gap on ScreenSpot-Pro.

**Variations**
1. Qwen2.5-VL-3B zero-shot (baseline from notebook 00)
2. Qwen2.5-VL-3B + LoRA (r ∈ {8, 16, 32, 64})
3. PaliGemma-3B + LoRA — set `BACKBONE = 'paligemma'` in the config cell to switch backbones

**Deeper analysis:** rank ablation (accuracy vs. trainable params vs. peak VRAM) + error analysis by UI type.


In [ ]:
# Colab bootstrap. On local Jupyter this is a no-op because the env is already set up.
import sys
from pathlib import Path

REPO_URL = "https://github.com/ali-epita/action-learning-ais5.git"
REPO_DIR = "/content/action-learning-ais5"

if "google.colab" in sys.modules:
    if not Path(REPO_DIR).exists():
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

    # --no-deps preserves Colab's pre-bundled torch + CUDA build.
    !pip install -q -e . --no-deps

    # transformers<5 because the Qwen2.5-VL wrapper targets 4.49+. peft for LoRA.
    !pip install -q "transformers>=4.49,<5" "accelerate>=0.34" "datasets>=3.0" \
                    "peft>=0.13" "qwen-vl-utils>=0.0.10" "bitsandbytes>=0.44" \
                    rich tqdm pyyaml typer pillow huggingface_hub safetensors psutil

    src_dir = str(Path(REPO_DIR) / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    import importlib
    importlib.invalidate_caches()

import ais5
print(f"ais5 v{ais5.__version__} ready  (colab={'google.colab' in sys.modules})")


In [ ]:
import json
from pathlib import Path

import pandas as pd

from ais5.adapt import LoRAConfig, TrainingArgs, run_lora_training
from ais5.data import load_benchmark
from ais5.eval import evaluate_model
from ais5.eval.breakdown import by_ui_type
from ais5.models import get_model
from ais5.utils import set_global_seed, setup_logging

setup_logging()
set_global_seed(42)


In [ ]:
# Backbone toggle — swap this to run the same rank sweep on PaliGemma.
# Both backbones share the same LoRA target-module names, dataset adapter,
# and notebook structure; only the prompt format and target-token convention
# differ (handled by the collator dispatch in ais5.adapt.make_collator).
BACKBONE = 'qwen2.5-vl'   # or 'paligemma'

if BACKBONE == 'qwen2.5-vl':
    BASE_MODEL = 'qwen2.5-vl-3b'
elif BACKBONE == 'paligemma':
    BASE_MODEL = 'paligemma-3b'
else:
    raise ValueError(f"Unknown BACKBONE {BACKBONE!r}")

# Smoke vs full toggle. SMOKE_MODE=True exercises every code path with one
# rank on a tiny subset — flip to False for the real H1 measurement.
SMOKE_MODE = True

if SMOKE_MODE:
    RANKS       = [8]
    TRAIN_N     = 500
    EVAL_LIMIT  = 50
    EPOCHS      = 1.0
    BENCHMARKS  = ['screenspot-v2']
else:
    RANKS       = [8, 16, 32, 64]
    TRAIN_N     = 50_000
    EVAL_LIMIT  = 200
    EPOCHS      = 1.0
    BENCHMARKS  = ['screenspot-v2', 'screenspot-pro', 'osworld-g']

TRAIN_DS     = 'OS-Copilot/OS-Atlas-data'
ADAPTER_NAME = 'os-atlas'             # row adapter in ais5.adapt.ROW_ADAPTERS

results_dir = Path('results/adapt')
results_dir.mkdir(parents=True, exist_ok=True)
jsonl_path = results_dir / f'sweep_{BACKBONE}.jsonl'
jsonl_path.unlink(missing_ok=True)

print(f"backbone={BACKBONE}  base_model={BASE_MODEL}")
print(f"SMOKE_MODE={SMOKE_MODE}  ranks={RANKS}  train_n={TRAIN_N}  benches={BENCHMARKS}")


## 1. Sanity check — single rank, single benchmark

Trains the first rank end-to-end on the smoke-sized subset, then evaluates on one benchmark. Once this converges, the sweep cell below just iterates over more ranks and benchmarks.


In [ ]:
first_rank = RANKS[0]
adapter_dir = Path(f'checkpoints/{BACKBONE}-r{first_rank}-{TRAIN_N}')

lora = LoRAConfig(r=first_rank, alpha=2 * first_rank, backbone=BACKBONE)
args = TrainingArgs(
    output_dir=str(adapter_dir),
    train_dataset=TRAIN_DS,
    adapter=ADAPTER_NAME,
    train_subset_size=TRAIN_N,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    bf16=True,
)
print(f"Training {BACKBONE} r={first_rank} on {TRAIN_N} samples ...")
adapter_dir = run_lora_training(BASE_MODEL, lora, args)
print(f"Saved adapter to {adapter_dir}")


## 2. Evaluate the first adapter


In [ ]:
model = get_model(BASE_MODEL, peft_adapter=str(adapter_dir))

first_eval = {}
for bench in BENCHMARKS:
    samples = list(load_benchmark(bench))
    run = evaluate_model(model, samples, benchmark=bench, limit=EVAL_LIMIT)
    first_eval[bench] = run
    print(f"{bench:18s} acc={run.accuracy:.4f}  (n={len(run.results)})")


## 3. Rank sweep

Trains each rank from scratch and evaluates on every benchmark in `BENCHMARKS`. Rows are streamed to `results/adapt/sweep_{BACKBONE}.jsonl` so a Colab disconnect doesn't lose accumulated progress. Re-running this cell will skip the first rank (already done in step 1) and continue with the rest.


In [ ]:
import torch

rows = []

# Seed with the first rank we already trained, so we don't re-run it.
already_done = {first_rank}
first_row = {
    'backbone': BACKBONE,
    'rank': first_rank,
    'train_n': TRAIN_N,
    'adapter_dir': str(adapter_dir),
}
for bench, run in first_eval.items():
    first_row[f'acc_{bench}'] = run.accuracy
    first_row[f'n_{bench}']   = len(run.results)
rows.append(first_row)
with jsonl_path.open('a') as f:
    f.write(json.dumps(first_row) + '\n')

# Free the first-rank model before training the next one.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

for rank in RANKS:
    if rank in already_done:
        continue
    ckpt = Path(f'checkpoints/{BACKBONE}-r{rank}-{TRAIN_N}')
    lora = LoRAConfig(r=rank, alpha=2 * rank, backbone=BACKBONE)
    args = TrainingArgs(
        output_dir=str(ckpt),
        train_dataset=TRAIN_DS,
        adapter=ADAPTER_NAME,
        train_subset_size=TRAIN_N,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=1e-4,
        bf16=True,
    )
    print(f"Training {BACKBONE} r={rank} ...")
    ckpt = run_lora_training(BASE_MODEL, lora, args)

    print(f"Evaluating r={rank} ...")
    m = get_model(BASE_MODEL, peft_adapter=str(ckpt))
    row = {'backbone': BACKBONE, 'rank': rank, 'train_n': TRAIN_N, 'adapter_dir': str(ckpt)}
    for bench in BENCHMARKS:
        samples = list(load_benchmark(bench))
        run = evaluate_model(m, samples, benchmark=bench, limit=EVAL_LIMIT)
        row[f'acc_{bench}'] = run.accuracy
        row[f'n_{bench}']   = len(run.results)
    rows.append(row)
    with jsonl_path.open('a') as f:
        f.write(json.dumps(row) + '\n')
    print(json.dumps(row, indent=2))

    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df = pd.DataFrame(rows).sort_values('rank').reset_index(drop=True)
df.to_csv(results_dir / f'sweep_{BACKBONE}.csv', index=False)
df


## 4. Error analysis by UI type

Where does LoRA help most? `by_ui_type` aggregates the per-sample correctness of the highest-rank adapter across the UI categories.


In [ ]:
ANALYSIS_RANK = max(RANKS)
analysis_ckpt = Path(f'checkpoints/{BACKBONE}-r{ANALYSIS_RANK}-{TRAIN_N}')
m = get_model(BASE_MODEL, peft_adapter=str(analysis_ckpt))

bench = BENCHMARKS[0]
samples = list(load_benchmark(bench))
full_run = evaluate_model(m, samples, benchmark=bench)
print(f"{BACKBONE} r={ANALYSIS_RANK} on {bench}: acc={full_run.accuracy:.4f}")
by_ui_type(full_run.results)
